# TinyChirp SincNet-Chunked (Axon-ready)

Single end-to-end SincNet model designed to fit the Axon NPU constraints (filter dims ≤ 16, strides ≤ 31, pool kernels ≤ 32, tensor dims ≤ 1024) and reproduce a clean "two parts at execution" structure:

1. **Per-chunk frontend** — SincNet convolution on each 1024-sample chunk (stride 8), followed by a cascaded avg-pool down to **one value per channel per chunk**.
2. **Head classifier** — Conv2D + cascaded global pool + dense over the sequence of chunk features.

Audio stays at 16 kHz (full 0–8 kHz band visible to the sinc filters). Total audio length is `46 × 1024 = 47104` samples (~2.94 s, fits inside the existing `TARGET_AUDIO_LEN_TIME`).

Self-contained: a local `SincnetConvW` is defined inline so the existing `model_parts.SincnetConv` (height-axis) is not touched.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import numpy as np
import tensorflow as tf

from utils import (
    TARGET_AUDIO_LEN_TIME,
    SAMPLE_RATE,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    make_time_datasets,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "sincnet_chunked"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
BATCH_SIZE = 32

## Chunked audio geometry

Input is shaped as `(NUM_CHUNKS, CHUNK_SIZE, 1)` NHWC: chunks live on the H axis, raw samples within a chunk live on W. A `SincnetConvW` kernel of shape `(1, k)` slides only along W, so each chunk row is processed independently with the same sinc filterbank — matches the deployment-time "frontend runs per chunk" semantics.

In [ ]:
CHUNK_SIZE = 1024              # 64 ms @ 16 kHz
NUM_CHUNKS = 46                # 46 * 1024 = 47104 samples ~ 2.94 s
TOTAL_LEN = NUM_CHUNKS * CHUNK_SIZE

SINCNET_FILTERS = 32
# SincNet needs an odd kernel for symmetric filters, and Axon's Conv2D filter
# dim is capped at 16 — so we use 15 (the nearest odd <= 16).
SINCNET_KERNEL = 15
SINCNET_STRIDE = 8             # user spec: 4 or 8

# Per-chunk pool cascade: 127 -> 7 -> 1   (each kernel <= 32)
CHUNK_POOL_1 = 16
CHUNK_POOL_2 = 7

# Head
HEAD_CONV_FILTERS = 16
HEAD_CONV_KERNEL = 5

# Inter-chunk pool cascade: 46 -> 2 -> 1  (each kernel <= 32)
CHUNK_SEQ_POOL_1 = 23
CHUNK_SEQ_POOL_2 = 2

In [ ]:
train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)
for x, y in train_ds.take(1):
    print("Dataset shape:", x.shape, "  labels:", y.shape)
print(f"Will crop {TARGET_AUDIO_LEN_TIME - TOTAL_LEN} samples from tail "
      f"and reshape to (NUM_CHUNKS={NUM_CHUNKS}, CHUNK_SIZE={CHUNK_SIZE}, 1)")

## SincNet layer with kernel on the **width** axis

Mirror of `model_parts.SincnetConv` but the filter has shape `(1, kernel_size, 1, num_filters)` and the conv strides along W. Each row (chunk) gets the same learnable bandpass filterbank — same kernel, processed independently.

In [ ]:
class SincnetConvW(tf.keras.layers.Layer):
    """SincNet on the width (W) axis. Input: (B, H, T, 1) -> (B, H, T_out, num_filters).

    Unlike model_parts.SincnetConv, kernel_size is honored as-is (no auto-bump
    to odd) so Axon's max filter dim of 16 stays respected.
    """

    def __init__(self, num_filters: int, kernel_size: int, stride: int,
                 sample_rate: int = 16000, **kwargs):
        super().__init__(**kwargs)
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.sample_rate = sample_rate

    def build(self, input_shape):
        mel_min = 0.0
        mel_max = 2595.0 * np.log10(1.0 + (self.sample_rate / 2.0) / 700.0)
        mel_points = np.linspace(mel_min, mel_max, self.num_filters + 1)
        hz_points = 700.0 * (10.0 ** (mel_points / 2595.0) - 1.0)
        f1_init = hz_points[:-1] / self.sample_rate
        band_init = np.diff(hz_points) / self.sample_rate

        self.f1 = self.add_weight(
            name="f1", shape=(self.num_filters,),
            initializer=tf.keras.initializers.Constant(f1_init), trainable=True,
        )
        self.band = self.add_weight(
            name="band", shape=(self.num_filters,),
            initializer=tf.keras.initializers.Constant(band_init), trainable=True,
        )
        # linspace from -(k//2) to k//2 with k samples. Odd k -> sample lands on t=0;
        # even k -> filter is centered between two taps (still well-defined).
        t = np.linspace(-(self.kernel_size // 2), self.kernel_size // 2, self.kernel_size)
        self.t = tf.constant(t, dtype=tf.float32)
        window = 0.54 - 0.46 * np.cos(
            2 * math.pi * np.arange(self.kernel_size) / (self.kernel_size - 1)
        )
        self.window = tf.constant(window, dtype=tf.float32)

    def get_filters(self) -> tf.Tensor:
        f1_safe = tf.math.abs(self.f1)
        f2_safe = f1_safe + tf.math.abs(self.band)
        f1_mat = tf.reshape(f1_safe, (1, -1))
        f2_mat = tf.reshape(f2_safe, (1, -1))
        t_mat = tf.reshape(self.t, (-1, 1))
        pi_t = math.pi * t_mat
        denom = tf.where(t_mat == 0.0, 1.0, pi_t)
        filters = (
            tf.math.sin(2.0 * math.pi * f2_mat * t_mat)
            - tf.math.sin(2.0 * math.pi * f1_mat * t_mat)
        ) / denom
        # Only triggered when a sample lands on t=0 (odd kernel). Harmless when even.
        center_values = 2.0 * (f2_mat - f1_mat)
        mask = tf.cast(t_mat == 0.0, tf.float32)
        filters = filters * (1.0 - mask) + center_values * mask
        filters = filters * tf.reshape(self.window, (-1, 1))
        return filters  # (kernel_size, num_filters)

    def get_filters_nhwc(self) -> tf.Tensor:
        # (k_h=1, k_w=kernel_size, in_c=1, out_c=num_filters)
        return tf.reshape(self.get_filters(), (1, self.kernel_size, 1, self.num_filters))

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        return tf.nn.conv2d(
            inputs,
            self.get_filters_nhwc(),
            strides=[1, 1, self.stride, 1],
            padding="VALID",
            data_format="NHWC",
        )

    def export_to_conv2d(self, name: str = "baked_sinc_conv_w") -> tf.keras.layers.Conv2D:
        baked = self.get_filters().numpy()
        w = np.reshape(baked, (1, self.kernel_size, 1, self.num_filters))
        conv_layer = tf.keras.layers.Conv2D(
            filters=self.num_filters,
            kernel_size=(1, self.kernel_size),
            strides=(1, self.stride),
            padding="valid",
            use_bias=False,
            name=name,
        )
        conv_layer.build(input_shape=(None, None, None, 1))
        conv_layer.set_weights([w])
        return conv_layer

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_filters": self.num_filters,
            "kernel_size": self.kernel_size,
            "stride": self.stride,
            "sample_rate": self.sample_rate,
        })
        return config

## Build the end-to-end training model

Per-chunk frontend: `SincnetConvW(k=16, s=8)` → ReLU → AvgPool(1,16) → AvgPool(1,7).
After this stage the shape is `(B, 46, 1, 32)` — exactly one feature per chunk per channel.

Head: Conv2D(5,1) padding=`same` → ReLU → AvgPool(23,1) → AvgPool(2,1) → Flatten → Dense(num_labels).

In [ ]:
from utils import get_flops_native


def build_training_model(num_labels: int) -> tf.keras.Model:
    # Match the standard time-domain dataset shape (B, TARGET_AUDIO_LEN_TIME, 1).
    inputs = tf.keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

    # Crop the tail to TOTAL_LEN = NUM_CHUNKS * CHUNK_SIZE, then reshape into chunks.
    tail_crop = TARGET_AUDIO_LEN_TIME - TOTAL_LEN
    x = tf.keras.layers.Cropping1D(cropping=(0, tail_crop), name="crop_to_chunks")(inputs)
    x = tf.keras.layers.Reshape((NUM_CHUNKS, CHUNK_SIZE, 1), name="chunkify")(x)

    # --- Per-chunk frontend ---
    x = SincnetConvW(
        num_filters=SINCNET_FILTERS,
        kernel_size=SINCNET_KERNEL,
        stride=SINCNET_STRIDE,
        sample_rate=SAMPLE_RATE,
        name="sincnet_w",
    )(x)
    x = tf.keras.layers.ReLU(name="sinc_relu")(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(1, CHUNK_POOL_1), strides=(1, CHUNK_POOL_1),
        padding="valid", name="chunk_pool_1",
    )(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(1, CHUNK_POOL_2), strides=(1, CHUNK_POOL_2),
        padding="valid", name="chunk_pool_2",
    )(x)
    # Shape here: (B, NUM_CHUNKS, 1, SINCNET_FILTERS) -> one value per chunk per filter.

    # --- Head classifier over the chunk sequence ---
    x = tf.keras.layers.Conv2D(
        filters=HEAD_CONV_FILTERS,
        kernel_size=(HEAD_CONV_KERNEL, 1),
        strides=(1, 1),
        padding="same",
        name="head_conv",
    )(x)
    x = tf.keras.layers.ReLU(name="head_relu")(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(CHUNK_SEQ_POOL_1, 1), strides=(CHUNK_SEQ_POOL_1, 1),
        padding="valid", name="head_pool_1",
    )(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(CHUNK_SEQ_POOL_2, 1), strides=(CHUNK_SEQ_POOL_2, 1),
        padding="valid", name="head_pool_2",
    )(x)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    outputs = tf.keras.layers.Dense(num_labels, activation=None, name="dense_logits")(x)

    return tf.keras.Model(inputs, outputs, name="sincnet_chunked_training")


training_model = build_training_model(num_labels)
training_model.summary()

flops = get_flops_native(training_model, batch_size=1)
print(f"Total FLOPs: {flops}")

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

training_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

init_wandb(MODEL_STEM, config={
    "chunk_size": CHUNK_SIZE,
    "num_chunks": NUM_CHUNKS,
    "sincnet_filters": SINCNET_FILTERS,
    "sincnet_kernel": SINCNET_KERNEL,
    "sincnet_stride": SINCNET_STRIDE,
    "head_conv_filters": HEAD_CONV_FILTERS,
    "head_conv_kernel": HEAD_CONV_KERNEL,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
})

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=get_callbacks(10, 5, BATCH_SIZE),
)
finish_wandb()

In [ ]:
from utils import plot_training_history
plot_training_history(history)

## Bake learned sinc filters into a static Conv2D

Replace the dynamic `SincnetConvW` with a frozen `Conv2D` of shape `(1, k, 1, num_filters)` so the int8 TFLite converter sees only standard ops.

In [ ]:
frontend_layer = training_model.get_layer("sincnet_w")
baked_conv = frontend_layer.export_to_conv2d(name="baked_sinc_conv_w")

infer_inputs = tf.keras.Input(shape=(NUM_CHUNKS, CHUNK_SIZE, 1))
x = baked_conv(infer_inputs)
for layer in training_model.layers:
    if isinstance(layer, (tf.keras.layers.InputLayer, type(frontend_layer))):
        continue
    x = layer(x)

inference_model = tf.keras.Model(infer_inputs, x, name="sincnet_chunked_inference")

for batch_audio, _ in test_ds.take(1):
    batch_np = batch_audio.numpy()
    logits_train = training_model.predict(batch_np, verbose=0)
    logits_infer = inference_model.predict(batch_np, verbose=0)
    print(f"Max abs diff: {np.max(np.abs(logits_train - logits_infer)):.8f}")

## Quantize and export INT8 TFLite

Representative samples are the same chunked tensors the model expects.

In [ ]:
rep_batches = build_representative_batches(test_ds, take=100)

try:
    export_keras_model_to_int8_tflite(inference_model, rep_batches, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"TFLite conversion failed: {e}")

In [ ]:
from utils import evaluate_tflite_model

hyperparams = {
    "chunk_size": CHUNK_SIZE,
    "num_chunks": NUM_CHUNKS,
    "sincnet_filters": SINCNET_FILTERS,
    "sincnet_kernel": SINCNET_KERNEL,
    "sincnet_stride": SINCNET_STRIDE,
    "head_conv_filters": HEAD_CONV_FILTERS,
    "head_conv_kernel": HEAD_CONV_KERNEL,
    "target_audio_len": TOTAL_LEN,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")